In [ ]:
import psutil
import os
from pydap.client import open_url
import numpy as np
import xarray as xr
import cftime
import xesmf as xe # Import the regridding library

# --- Installation Note ---
# xesmf is best installed via conda to handle its complex dependencies:
# conda install -c conda-forge xesmf esmpy

In [ ]:
# --- Part 1: Data Loading (Your original code) ---

# Track network usage before download
# Open dataset from OPeNDAP using pydap
url = "dap2://tds.vims.edu:8080/thredds/dodsC/abever/ECB_FORECAST_HR/AVERAGES/chesroms_ECB_HR_avg_20250812.nc"
dataset = open_url(url)

print(dataset)
print(dataset['ocean_time'].attributes)

In [ ]:
# Extract variables as numpy arrays
oxygen_np = np.array(dataset['oxygen'])      # shape (time, s_rho, eta_rho, xi_rho)
salt_np = np.array(dataset['salt'])          # shape (time, s_rho, eta_rho, xi_rho)
lon_rho_np = np.array(dataset['lon_rho'])    # shape (eta_rho, xi_rho)
lat_rho_np = np.array(dataset['lat_rho'])    # shape (eta_rho, xi_rho)
ocean_time_np = np.array(dataset['ocean_time'])

# Convert ocean_time to datetime objects
time_units = dataset['ocean_time'].attributes.get('units', 'seconds since 2009-01-01 00:00:00')
calendar = dataset['ocean_time'].attributes.get('calendar', 'proleptic_gregorian')
time_dt = cftime.num2date(ocean_time_np, units=time_units, calendar=calendar)

# Select surface layer (highest s_rho index, which is -1)
oxygen_surface_np = oxygen_np[:, -1, :, :]  # shape (time, eta_rho, xi_rho)
salt_surface_np = salt_np[:, -1, :, :]

print("Original oxygen surface shape:", oxygen_surface_np.shape)

In [ ]:
# Create an xarray Dataset from the numpy arrays for regridding
# This is the dataset we will use as the *source* for regridding.
ds_source_2d = xr.Dataset(
    {
        "oxygen_surface": (("time", "eta_rho", "xi_rho"), oxygen_surface_np, dataset['oxygen'].attributes),
        "salt_surface": (("time", "eta_rho", "xi_rho"), salt_surface_np, dataset['salt'].attributes),
    },
    # xesmf expects coordinates to be named 'lon' and 'lat'
    coords={
        "lon": (("eta_rho", "xi_rho"), lon_rho_np),
        "lat": (("eta_rho", "xi_rho"), lat_rho_np),
        "time": time_dt,
    },
    attrs=dataset.attributes
)

print("\n--- Original Dataset on Curvilinear Grid ---")
print(ds_source_2d)

In [ ]:
# --- Part 2: Regridding to WGS84 ---

# 1. Define the target WGS84 grid.
# We'll create a regular grid with a specified resolution (e.g., 0.02 degrees).
# The grid will cover the extent of the original data.
grid_resolution = 0.005
target_lon = np.arange(lon_rho_np.min(), lon_rho_np.max(), grid_resolution)
target_lat = np.arange(lat_rho_np.min(), lat_rho_np.max(), grid_resolution)
ds_target = {"lon": target_lon, "lat": target_lat}

print("\n--- Target WGS84 Grid Definition ---")
print(f"Target lon shape: {target_lon.shape}, Target lat shape: {target_lat.shape}")


print("\n--- Target WGS84 Grid Definition ---")
print(ds_target)


# 2. Create the regridder object.
# This object calculates the weights needed to map from the source to the target grid.
# 'bilinear' is a good interpolation method for continuous data like salt and oxygen.
# reuse_weights=True allows you to save and reload weights for faster future calculations.
regridder = xe.Regridder(ds_source_2d, ds_target, "nearest_s2d")
print("\n--- Regridder Created ---")
print(regridder)


# 3. Apply the regridding.
# The regridder object is called like a function on the source dataset.
ds_regridded = regridder(ds_source_2d, keep_attrs=True, skipna=True)
print("\n--- Regridded Dataset on WGS84 Grid ---")
print(ds_regridded)

In [ ]:
ds_regridded.to_netcdf('regridded' + os.path.basename(url))